# Jealous Couples

Three couples need to cross a river.  They have to follow the following rules: 
- The boat can only carry two persons.
- At least one person has to be on the boat on every crossing.
- The husbands know their wives well and would never leave them with another man, unless they are present themselves.

We are going to formulate this problem as a *symbolic transition system*.
Then, we can solve the problem using the constraint solver `Z3`.

We assume the first couple is called `Anton` and `Doris`, the names of the second couple are
`Brian` and `Eliza`, while the last couple is called `Charly` and `Freya`.  Hence we
will use the following variables to encode the problem:
* `A` equals `1` if `Anton` is on the western shore,
* `B` equals `1` if `Brian` is on the western shore,
* `C` equals `1` if `Charly` is on the western shore,
* `D` equals `1` if `Doris` is on the western shore,
* `E` equals `1` if `Eliza` is on the western shore,
* `F` equals `1` if `Freya` is on the western shore,
* `S` equals `1` if the ship is on the western shore.

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

In [3]:
let A = Int "A"
let B = Int "B"
let C = Int "C"
let D = Int "D"
let E = Int "E"
let F = Int "F"
let S = Int "S"

The function `start` takes the seven variables describing a state.  It returns a list of constraints that describe the initial state.

In [4]:
let start (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (E: IntExpr) (F: IntExpr) (S: IntExpr) : BoolExpr list =
    [ A .= 1; B .= 1; C .= 1; D .= 1; E .= 1; F .= 1; S .= 1 ]

In [5]:
start A B C D E F S

[A = 1; B = 1; C = 1; D = 1; E = 1; F = 1; S = 1]

The function `goal` takes the seven variables describing a state.  It returns a list of constraints that describe the final state.

In [6]:
let goal (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (E: IntExpr) (F: IntExpr) (S: IntExpr) : BoolExpr list =
    [ A .= 0; B .= 0; C .= 0; D .= 0; E .= 0; F .= 0; S .= 0 ]

In [7]:
goal A B C D E F S

[A = 0; B = 0; C = 0; D = 0; E = 0; F = 0; S = 0]

The function `invariant` takes the seven variables describing a state.  It returns a list of formulas that need to be true after every transition:  If a wife is not on the same shore as her husband, then she must not be on the same shore as one of the other husbands.

In [8]:
let invariant (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (E: IntExpr) (F: IntExpr) (S: IntExpr) : BoolExpr list =
    [ Implies(D .<> A, And [ D .<> B; D .<> C ])
      Implies(E .<> B, And [ E .<> A; E .<> C ])
      Implies(F .<> C, And [ F .<> A; F .<> B ]) ]

In [9]:
invariant A B C D E F S

[¬(D = A) → ¬(D = B) ∧ ¬(D = C); ¬(E = B) → ¬(E = A) ∧ ¬(E = C); ¬(F = C) → ¬(F = A) ∧ ¬(F = B)]

The function `transition` takes fourteen arguments:  the variables `A`, $\cdots$, `S` describe the state before the crossing, while `Ax`, $\cdots$, `Sx` describe the state after the crossing.  It returns a list of formulas that describe the transitions:
- If the ship is on the western shore, one or two persons travel from the western shore to the eastern shore.
- If the ship is on the eastern shore, one or two persons travel from the eastern shore to the western shore.
- The ship changes sides.

In [10]:
let transition (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (E: IntExpr) (F: IntExpr) (S: IntExpr)
               (Ax: IntExpr) (Bx: IntExpr) (Cx: IntExpr) (Dx: IntExpr) (Ex: IntExpr) (Fx: IntExpr) (Sx: IntExpr)
               : BoolExpr list =
    [ Implies(S .= 1,
              And [ A .>= Ax; B .>= Bx; C .>= Cx; D .>= Dx; E .>= Ex; F .>= Fx
                    (A+B+C+D+E+F) - (Ax+Bx+Cx+Dx+Ex+Fx) .<= 2
                    (A+B+C+D+E+F) - (Ax+Bx+Cx+Dx+Ex+Fx) .>= 1 ])
      Implies(S .= 0, A .<= Ax)
      Implies(S .= 0, B .<= Bx)
      Implies(S .= 0, C .<= Cx)
      Implies(S .= 0, D .<= Dx)
      Implies(S .= 0, E .<= Ex)
      Implies(S .= 0, F .<= Fx)
      Implies(S .= 0, (Ax+Bx+Cx+Dx+Ex+Fx) - (A+B+C+D+E+F) .<= 2)
      Implies(S .= 0, (Ax+Bx+Cx+Dx+Ex+Fx) - (A+B+C+D+E+F) .>= 1)
      Sx + S .= 1 ]

The function `jealousCSP` takes one argument:
- `n` is the number of crossings.

It tries to solve the problem with `n` crossings.  If this is possible, `Some Solution` is returned, where `Solution` is a map assigning values to the variables.  Otherwise, `None` is returned.

In [11]:
let jealousCSP (n: int) : Map<string, int> option =
    let Solver = newSolver ()
    let vars name = [| for i in 0 .. n -> Int (sprintf "%s%d" name i) |]
    let As, Bs, Cs, Ds, Es, Fs, Ss = vars "A", vars "B", vars "C", vars "D", vars "E", vars "F", vars "S"
    let Cts = [ yield! start As[0] Bs[0] Cs[0] Ds[0] Es[0] Fs[0] Ss[0]
                yield! goal  As[n] Bs[n] Cs[n] Ds[n] Es[n] Fs[n] Ss[n]
                for i in 0 .. n - 1 do
                    let j = i + 1
                    yield! invariant  As[i] Bs[i] Cs[i] Ds[i] Es[i] Fs[i] Ss[i]
                    yield! transition As[i] Bs[i] Cs[i] Ds[i] Es[i] Fs[i] Ss[i]
                                      As[j] Bs[j] Cs[j] Ds[j] Es[j] Fs[j] Ss[j]
                    for X in [ As; Bs; Cs; Ds; Es; Fs; Ss ] do
                        yield 0 .<= X[i]
                        yield X[i] .<= 1 ]
    Solver.Add(Array.ofList Cts)
    if Solver.Check() = Status.SATISFIABLE then
        let m = Solver.Model
        Some (Map [ for (name, X) in [ ("A", As); ("B", Bs); ("C", Cs); ("D", Ds); ("E", Es); ("F", Fs) ] do
                      for i in 0 .. n -> (sprintf "%s%d" name i, intValue m X[i]) ])
    else
        None

In [12]:
jealousCSP 3

None

The function `findSolution` takes no arguments.  It tries $n = 1, 3, 5, \cdots$ crossings until a solution is found.  It returns the number of crossings together with the solution.

In [13]:
let findSolution () : int * Map<string, int> =
    let rec search (n: int) =
        printfn "%d" n
        match jealousCSP n with
        | Some solution -> (n, solution)
        | None          -> search (n + 2)
    search 1

In [14]:
#!time
let n, Solution = findSolution ()

1

3

5

7

9

11

Wall time: 88.5692ms

In [15]:
n, Solution

(11, map [("A0", 1); ("A1", 1); ("A10", 0); ("A11", 0); ("A2", 1); ("A3", 1); ("A4", 1); ("A5", 0); ("A6", 1); ("A7", 0); ("A8", 0); ("A9", 0); ("B0", 1); ("B1", 1); ("B10", 0); ("B11", 0); ("B2", 1); ("B3", 1); ("B4", 1); ("B5", 1); ("B6", 1); ("B7", 0); ("B8", 0); ("B9", 0); ("C0", 1); ("C1", 1); ("C10", 1); ("C11", 0); ("C2", 1); ("C3", 1); ("C4", 1); ("C5", 0); ("C6", 0); ("C7", 0); ("C8", 0); ("C9", 0); ("D0", 1); ("D1", 0); ("D10", 0); ("D11", 0); ("D2", 0); ("D3", 0); ("D4", 0); ("D5", 0); ("D6", 1); ("D7", 1); ("D8", 1); ("D9", 0); ("E0", 1); ("E1", 1); ("E10", 0); ("E11", 0); ("E2", 1); ("E3", 0); ("E4", 1); ("E5", 1); ("E6", 1); ("E7", 1); ("E8", 1); ("E9", 0); ("F0", 1); ("F1", 0); ("F10", 1); ("F11", 0); ("F2", 1); ("F3", 0); ("F4", 0); ("F5", 0); ("F6", 0); ("F7", 0); ("F8", 1); ("F9", 1)])

## Auxiliary Code for Pretty Printing

The function `showSolution` takes two arguments:
- `Solution` is a map assigning values to the variables,
- `n` is the number of crossings.

It prints the path that has been found.  We won't discuss the details of this function.

In [16]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    let get name i = Solution[sprintf "%s%d" name i]
    let persons A B C D E F = rep A "👨" + rep B "👨🏼" + rep C "👨🏿" + rep D "👩" + rep E "👩🏼" + rep F "👩🏿"
    for i in 0 .. n do
        let A, B, C, D, E, F = get "A" i, get "B" i, get "C" i, get "D" i, get "E" i, get "F" i
        printfn "%s%s%s" (persons A B C D E F) (String.replicate 42 " ") (persons (1-A) (1-B) (1-C) (1-D) (1-E) (1-F))
        if i < n then
            let sign = if i % 2 = 0 then 1 else -1
            let crossing name = sign * (get name i - get name (i+1))
            let arrows = if i % 2 = 0 then ">>>" else "<<<"
            printfn "%s%s%s%s" (String.replicate 24 " ") arrows
                    (persons (crossing "A") (crossing "B") (crossing "C") (crossing "D") (crossing "E") (crossing "F")) arrows

In [17]:
showSolution Solution n

👨👨🏼👨🏿👩👩🏼👩🏿

>>>

👩👩🏿

>>>

👨👨🏼👨🏿👩🏼

...

👨👨🏼👩👩🏼

>>>

👨🏿👩🏿

>>>

👨👨🏼👨🏿👩👩🏼👩🏿